<a href="https://colab.research.google.com/github/MeganMulholland/MBTIPredictor/blob/main/FinalMBTI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MBTI Personality Prediction
**Project Goal:** The project aims to predict a person's MBTI personality type based on their text posts using Natural Language Processing (NLP) and machine learning.

**Steps**:

1.   Data Preparation:
  Imported the MBTI dataset (mbti_1.csv).
  Cleaned the text data by removing links and MBTI type mentions.
2.  Data Preprocessing:
  Converted MBTI types into numerical labels using Label Encoding.
Split the data into training and validation sets.
Tokenized the text data using the distilbert-base-uncased tokenizer from Hugging Face Transformers.
3. Model Building and Training:
Used a pre-trained DistilBERT model (distilbert-base-uncased) for sequence classification and fine-tuned it for MBTI prediction.
Trained the model using the Hugging Face Trainer with specified hyperparameters.
Evaluated the model's performance using accuracy as the metric.
4. Model Evaluation and Visualization:
Plotted the validation accuracy and loss over epochs to visualize the training process.
Generated a classification report and confusion matrix to assess the model's performance on different MBTI types.
Visualized per-class precision, recall, and F1-score using a bar chart.
5. Binary MBTI Trait Prediction:
Extracted binary features for each MBTI trait (I/E, N/S, T/F, J/P).
Trained separate models for each trait using the BERT model.
Evaluated the accuracy of each model.
Generated classification reports and confusion matrices for each trait.
Compared the accuracy of all binary trait models.

**Key Libraries:**

pandas, re, nltk, sklearn, transformers, datasets, evaluate, torch, matplotlib, seaborn

In [ ]:
!pip install nltk==3.8.1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 1. Data Preparation and 2. Preprocessing

In [ ]:
import pandas as pd
import re
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
mbti_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/mbti_1.csv')
mbti_data.head()

In [ ]:
def remove_links(text):
  return re.sub(r'http\S+', '', text)

mbti_data['posts'] = mbti_data['posts'].apply(remove_links)
mbti_data.head()

In [ ]:
def remove_mbti_mentions(text):
  mbti_types = ["INFP", "INFJ", "INTP", "INTJ", "ENTP", "ENTJ", "ENFP", "ENFJ",
                "ISFP", "ISFJ", "ISTP", "ISTJ", "ESFP", "ESFJ", "ESTP", "ESTJ"]

  pattern = r'\b.?(?:' + '|'.join(mbti_types) + r')s?.?\b'
  return re.sub(pattern, '', text, flags=re.IGNORECASE)

mbti_data['posts'] = mbti_data['posts'].apply(remove_mbti_mentions)
mbti_data.head()

# 3. Model Building and Training

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn joblib

In [ ]:
!pip uninstall -y transformers huggingface_hub accelerate
!pip install --no-cache-dir transformers \
                             huggingface_hub \
                             accelerate \
                             datasets evaluate scikit-learn joblib

!pip install hf_xet

This is the first model we created using the Distilbert model, which is smaller. Using the BERT model originally caused overfitting issues, so we changed the model used to the Distilbert model which is smaller. This prevented the model from memorizing the data, works faster, and our data only contains 8,600 entries so BERT is not technically necessary.

In [ ]:
print(mbti_data["type"].value_counts())   # See class balance
print(mbti_data.isna().sum())             # Make sure no NaNs in posts / type

# 3.  Encode labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
mbti_data["label"] = le.fit_transform(mbti_data["type"])   # 0-15

# 4.  Train / validation split
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    mbti_data[["posts", "label"]],
    test_size=0.10,
    stratify=mbti_data["label"],
    random_state=42
)

# 5.  Hugging Face Datasets & tokeniser

from datasets import Dataset
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))

from transformers import AutoTokenizer
#MODEL_NAME = "bert-base-uncased" larger model
MODEL_NAME = "distilbert-base-uncased"  #smaller model, better to prevent overfitting, will not memorize data and our data set is under 10,000 entries
MAX_LEN = 256

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tok(batch["posts"], truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["posts"])
val_ds   = val_ds.map(tokenize,   batched=True, remove_columns=["posts"])

# 6.  Build the model

from transformers import AutoModelForSequenceClassification
num_labels = len(le.classes_)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)



In [ ]:

from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate, torch

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.argmax(torch.tensor(logits), dim=-1)
    return accuracy.compute(predictions=preds, references=labels)

args = TrainingArguments(
    eval_strategy="epoch",
    save_strategy="epoch",
    output_dir="bert_mbti",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    num_train_epochs=5,
    #evaluation_strategy="epoch",
    #save_strategy="epoch",
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(),    # mixed precision on GPU
    logging_steps=50,
    report_to = "none"
)
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tok,
    data_collator=DataCollatorWithPadding(tok),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  #will stop training if validation accuracy doesn't improve for 2 epochs
)

# 8.  Train & evaluate

trainer.train()
print(trainer.evaluate())

# 9.  Save artefacts
trainer.save_model("bert_mbti/best")
tok.save_pretrained("bert_mbti/best")

import joblib
joblib.dump(le, "bert_mbti/label_encoder.pkl")

#
# 10.  Handy inference function
def predict_mbti(raw_text: str, model_path="bert_mbti/best"):
    tok   = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    le    = joblib.load(f"{model_path}/../label_encoder.pkl")

    inputs = tok(raw_text, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = logits.argmax(-1).item()
    return le.inverse_transform([pred])[0]

# Quick test
#print(predict_mbti("I love abstract ideas and strategic planning."))  # → e.g. "INTJ

In [ ]:
trainer.state.log_history  # Shows list of logged metrics


In [ ]:

import matplotlib.pyplot as plt
log_df = pd.DataFrame(trainer.state.log_history)
eval_df = log_df.dropna(subset=["eval_loss", "eval_accuracy"])
plt.plot(eval_df["epoch"], eval_df["eval_accuracy"], marker="o", label="Val Accuracy")
plt.plot(eval_df["epoch"], eval_df["eval_loss"],       marker="o", label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Metric")
plt.legend()
plt.title("Validation Accuracy & Loss over Epochs")
plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Get predictions
preds_output = trainer.predict(val_ds)
y_pred = np.argmax(preds_output.predictions, axis=1)
y_true = preds_output.label_ids

print(classification_report(y_true, y_pred, target_names=le.classes_))


In [ ]:
# confusion matrix heat map
import itertools


# 1. Compute
cm = confusion_matrix(y_true, y_pred)

# 2. Plot
plt.figure(figsize=(12,10))
plt.imshow(cm, interpolation='nearest')       # default colormap
plt.title("Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")

# 3. Tick labels
classes = le.classes_
plt.xticks(np.arange(len(classes)), classes, rotation=90)
plt.yticks(np.arange(len(classes)), classes)

# 4. Annotate each cell with its count
thresh = cm.max() / 2
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(
        j, i, cm[i, j],
        ha="center", va="center",
        color="white" if cm[i, j] > thresh else "black",
        fontsize=6
    )

plt.tight_layout()
plt.show()


In [ ]:
#Precision/Recall/F1
from sklearn.metrics import classification_report

# 1. Get dict form of the report
report_dict = classification_report(
    y_true,
    y_pred,
    target_names=le.classes_,
    output_dict=True
)
#build df
report_df = pd.DataFrame(report_dict).T
metrics_df = report_df.loc[le.classes_, ["precision", "recall", "f1-score"]]

# 3. Plot
metrics_df.plot(kind="bar", figsize=(14,6))
plt.title("Per-Class Precision, Recall & F1-Score")
plt.xlabel("MBTI Type")
plt.ylabel("Score")
plt.xticks(rotation=45, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:
mbti_data1 = mbti_data.copy()

In [ ]:
# Extract I/E, N/S, T/F, J/P from MBTI types and create separate columns
mbti_data1['I/E'] = mbti_data1['type'].apply(lambda x: 1 if x[0] == 'I' else 0)  # I/E
mbti_data1['N/S'] = mbti_data1['type'].apply(lambda x: 1 if x[1] == 'N' else 0)  # N/S
mbti_data1['T/F'] = mbti_data1['type'].apply(lambda x: 1 if x[2] == 'T' else 0)  # T/F
mbti_data1['J/P'] = mbti_data1['type'].apply(lambda x: 1 if x[3] == 'J' else 0)  # J/P


In [ ]:
mbti_data1.head()

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate

# Load tokenizer
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# STEP 1: Extract binary MBTI columns
mbti_data1['I/E'] = mbti_data1['type'].apply(lambda x: 1 if x[0] == 'I' else 0)
mbti_data1['N/S'] = mbti_data1['type'].apply(lambda x: 1 if x[1] == 'N' else 0)
mbti_data1['T/F'] = mbti_data1['type'].apply(lambda x: 1 if x[2] == 'T' else 0)
mbti_data1['J/P'] = mbti_data1['type'].apply(lambda x: 1 if x[3] == 'J' else 0)

# STEP 2: Function to create tokenized train and val datasets
def prepare_mbti_binary_datasets(df, label_col, test_size=0.1):
    train_df, val_df = train_test_split(df[['posts', label_col]], test_size=test_size, stratify=df[label_col], random_state=42)

    train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
    val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))

    def tokenize(batch):
        return tokenizer(batch["posts"], truncation=True, padding="max_length", max_length=MAX_LEN)

    train_ds = train_ds.map(tokenize, batched=True, remove_columns=["posts"])
    val_ds = val_ds.map(tokenize, batched=True, remove_columns=["posts"])

    train_ds = train_ds.rename_column(label_col, "labels")
    val_ds = val_ds.rename_column(label_col, "labels")

    train_ds.set_format("torch")
    val_ds.set_format("torch")

    return train_ds, val_ds

# STEP 3: Model training function
def train_model(train_dataset, val_dataset, task_name, num_labels=2):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

    args = TrainingArguments(
        output_dir=f"bert_{task_name}",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        learning_rate=2e-5,
        logging_dir=f"logs_{task_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        report_to="none",
    )

    accuracy = evaluate.load("accuracy")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = torch.argmax(torch.tensor(logits), dim=-1)
        return accuracy.compute(predictions=preds, references=labels)

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    return trainer

# STEP 4: Prepare datasets for all 4 MBTI traits
train_ds_ie, val_ds_ie = prepare_mbti_binary_datasets(mbti_data1, "I/E")
train_ds_ns, val_ds_ns = prepare_mbti_binary_datasets(mbti_data1, "N/S")
train_ds_tf, val_ds_tf = prepare_mbti_binary_datasets(mbti_data1, "T/F")
train_ds_jp, val_ds_jp = prepare_mbti_binary_datasets(mbti_data1, "J/P")

# STEP 5: Train models
trainer_ie = train_model(train_ds_ie, val_ds_ie, "IE")
trainer_ns = train_model(train_ds_ns, val_ds_ns, "NS")
trainer_tf = train_model(train_ds_tf, val_ds_tf, "TF")
trainer_jp = train_model(train_ds_jp, val_ds_jp, "JP")



In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

def evaluate_and_plot(trainer, val_dataset, title):
    preds_output = trainer.predict(val_dataset)
    y_pred = np.argmax(preds_output.predictions, axis=1)
    y_true = preds_output.label_ids

    # Print classification report
    print(f"\nClassification Report for {title}:")
    print(classification_report(y_true, y_pred, target_names=[f"Not {title}", title], zero_division=0))


    # Plot confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[f"Not {title}", title])
    disp.plot(cmap='Blues')
    plt.title(f"{title} Confusion Matrix")
    plt.show()

# Evaluate each model
evaluate_and_plot(trainer_ie, val_ds_ie, "Introvert (I)")
evaluate_and_plot(trainer_ns, val_ds_ns, "Intuitive (N)")
evaluate_and_plot(trainer_tf, val_ds_tf, "Thinking (T)")
evaluate_and_plot(trainer_jp, val_ds_jp, "Judging (J)")


In [ ]:
def get_accuracy(trainer, val_dataset):
    preds_output = trainer.predict(val_dataset)
    y_pred = np.argmax(preds_output.predictions, axis=1)
    y_true = preds_output.label_ids
    return np.mean(y_pred == y_true)

accuracies = {
    "I/E": get_accuracy(trainer_ie, val_ds_ie),
    "N/S": get_accuracy(trainer_ns, val_ds_ns),
    "T/F": get_accuracy(trainer_tf, val_ds_tf),
    "J/P": get_accuracy(trainer_jp, val_ds_jp)
}

# Plot bar chart
plt.figure(figsize=(8, 5))
sns.barplot(x=list(accuracies.keys()), y=list(accuracies.values()))
plt.ylim(0, 1)
plt.ylabel("Accuracy")
plt.title("Accuracy per MBTI Trait")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

def run_tfidf_baseline(df, target, test_size=0.2, random_state=42):
    """Return accuracy and fitted pipeline for one MBTI binary target."""
    X_train, X_val, y_train, y_val = train_test_split(
        df["posts"], df[target],
        test_size=test_size, stratify=df[target], random_state=random_state
    )

    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),        # unigrams + bigrams
            max_features=30_000
        )),
        ("clf", LogisticRegression(
            max_iter=2_000,
            n_jobs=-1,                 # speed-up on multi-core
            solver="lbfgs"
        ))
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)

    # Metrics
    acc = accuracy_score(y_val, y_pred)
    print(f"\n=== {target}  |  TF-IDF baseline ===")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_val, y_pred, digits=3))

    # Confusion matrix
    ConfusionMatrixDisplay(
        confusion_matrix(y_val, y_pred),
        display_labels=["0", "1"]
    ).plot(cmap="Blues", values_format="d")
    plt.title(f"{target} – TF-IDF baseline")
    plt.show()

    return acc, pipe


tfidf_results = {}
for col in ["I/E", "N/S", "T/F", "J/P"]:
    tfidf_results[col], _ = run_tfidf_baseline(mbti_data1, col)

#
# Compact scoreboard versus your fine-tuned BERT models
# stored BERT validation accuracy in the dict `bert_acc
bert_acc = {
    "I/E": trainer_ie.evaluate()["eval_accuracy"],
    "N/S": trainer_ns.evaluate()["eval_accuracy"],
    "T/F": trainer_tf.evaluate()["eval_accuracy"],
    "J/P": trainer_jp.evaluate()["eval_accuracy"],
}

import pandas as pd
pd.DataFrame({"BERT": bert_acc, "TF-IDF": tfidf_results})